In [1]:
import numpy as np
import random

# 0=空地, 1=障碍, 2=终点
grid = np.array([
    [0, 0, 0, 0, 0],
    [0, 1, 1, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 1, 0, 1, 0],
    [0, 0, 0, 0, 2],
])

start = (0, 0)
goal = (4, 4)

actions = {
    0: (-1, 0),  # 上
    1: (1, 0),   # 下
    2: (0, -1),  # 左
    3: (0, 1),   # 右
}

num_states = grid.shape[0] * grid.shape[1]
num_actions = 4

Q = np.zeros((num_states, num_actions))

alpha = 0.1      # 学习率
gamma = 0.9      # 折扣因子
epsilon = 0.2    # 探索概率
episodes = 2000


def state_to_id(pos):
    r, c = pos
    return r * grid.shape[1] + c


def step(pos, action):
    r, c = pos
    dr, dc = actions[action]

    nr, nc = r + dr, c + dc

    # 撞墙
    if nr < 0 or nr >= 5 or nc < 0 or nc >= 5:
        return pos, -5, False

    # 撞障碍
    if grid[nr, nc] == 1:
        return pos, -5, False

    # 到达终点
    if (nr, nc) == goal:
        return (nr, nc), 100, True

    # 普通移动
    return (nr, nc), -1, False


for episode in range(episodes):
    pos = start
    done = False

    while not done:
        s = state_to_id(pos)

        # epsilon-greedy
        if random.random() < epsilon:
            action = random.randint(0, num_actions - 1)
        else:
            action = np.argmax(Q[s])

        next_pos, reward, done = step(pos, action)
        ns = state_to_id(next_pos)

        # Q-learning 更新公式
        # 旧经验+新走一步得到的反馈=更新后的经验
        Q[s, action] = Q[s, action] + alpha * (
            reward + gamma * np.max(Q[ns]) - Q[s, action]
        )

        pos = next_pos


# 打印训练后的最优路径
pos = start
path = [pos]

for _ in range(50):
    s = state_to_id(pos)
    action = np.argmax(Q[s])
    next_pos, reward, done = step(pos, action)
    path.append(next_pos)
    pos = next_pos

    if done:
        break

print("最优路径：")
print(path)

最优路径：
[(0, 0), (0, 1), (0, 2), (0, 3), (1, 3), (1, 4), (2, 4), (3, 4), (4, 4)]
